# Claim Diagnosis Features — All Cohorts

Cohort-based refactor of `03_claim_diagnosis_monthly` (NBA 3.0, Wenjie Chen). Feature logic is unchanged: CKD stage of each Dx claim (`TOTAL_DX_CLAIMS_STAGE_*`), stage the patient is new to in that month (`TOTAL_DX_CLAIMS_WITH_PATIENT_*`), total Dx claims, `RUNNING_` (full history) and `PCT_` features over the original lookbacks **1, 3, 6, 12 months**. Only the time grain changes: instead of calendar months (`SVC_DT_M` + row windows), every cohort in `00_cohort_config` gets date-range windows ending at its cutoff (cohort month end + `OFFSET_DAY` − 1), and one row per NPI per `COHORT_ID` is written to `claim_diagnosis_monthly_cohort` and `provider_claim_diagnosis_monthly_cohort`.

Source: `/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/03_Feature_Engineering/03_HCP_Level_Features/03_claim_diagnosis_monthly`

## 0. Connection and Config

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "../00_cohort_config"

In [0]:
# All config (OFFSET_DAYS, COHORT_MONTHS, NPI_COL, SVC_COL, VALIDATION_MODE, get_cohort_months,
# get_cohort_bounds, build_feature_group, assign_cohort_month, apply_npi_filter, ...) comes from
# 00_cohort_config. 00_config / 00_step23_function are no longer needed.

# Original lookbacks of this notebook (last_i_month_list) — no 9-month window, unlike NB01/NB02
DX_LOOKBACK_WINDOWS = [1, 3, 6, 12]

_cohort_setting = (COHORT_MONTHS if COHORT_MONTHS is not None
                   else f"all {COHORT_ORIGIN[:7]} .. {COHORT_LAST[:7]} (complete data only)")
print(f"Offset: {OFFSET_DAYS} | Cohorts: {_cohort_setting} | Lookbacks: {DX_LOOKBACK_WINDOWS} | "
      f"Validation: {VALIDATION_MODE}")

## 1. Query Data

In [0]:
import builtins
_QUERY_OUTPUT = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/"
                 "kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/")

df_dx_raw = spark.read.format("delta").load(_QUERY_OUTPUT + "Dx_03")

# Month truncation (date_trunc('MM', SVC_DT)) removed: daily SVC_DT is kept and windows are
# date ranges per cohort. DX_STAGE label mapping is the same as the original 2.1 cell.
def add_dx_stage_label(df):
    return df.withColumn("DX_STAGE",
        F.when((F.col("DX_STAGE") == 1), "STAGE_1").otherwise(
        F.when((F.col("DX_STAGE") == 2), "STAGE_2").otherwise(
        F.when((F.col("DX_STAGE") == 3), "STAGE_3").otherwise(
        F.when((F.col("DX_STAGE") == 4), "STAGE_4").otherwise(
        F.when((F.col("DX_STAGE") == 5), "STAGE_5").otherwise("STAGE_UNSPECIFIED"))))))

DATA_MAX_DT = df_dx_raw.agg(F.max(SVC_COL)).first()[0]

# Validation mode: restrict to the 10 test HCPs (no-op when VALIDATION_MODE = False)
df_dx = add_dx_stage_label(apply_npi_filter(df_dx_raw, NPI_COL))

# ---------- Patient CKD stage (stage the patient is new to, per month) ----------
# Monthly table from the patient journey, keyed on offset-aligned SVC_DT_M.
# Validation mode reads the 3.1 validation output (to be rebuilt at the current OFFSET_DAY before
# offset > 0 results are final). Production still reads the 3.0 table until a 3.1 output exists.
_CKD_TABLE = ("kerendia_nba_3_1_pat_new_ckd_stage_validation" if VALIDATION_MODE
              else "pat_new_ckd_stage")
pat_new_ckd_stage = spark.read.format("delta").load(_QUERY_OUTPUT + _CKD_TABLE)
print(f"Patient CKD stage: {_CKD_TABLE}")

def add_patient_stage(df, offset_days, pat_df=None):
    """Original 2.2 join (claim month = patient-stage month) at claim level. The claim month is
    assign_cohort_month(SVC_DT, offset_days) instead of date_trunc('MM'); same at offset 0.
    Kept separate from df_dx, as in the original (only the patient-stage pivot uses it).
    pat_df: patient-stage table to use (default pat_new_ckd_stage)."""
    _pat = ((pat_new_ckd_stage if pat_df is None else pat_df)
            .select(F.col("PATIENT_ID").alias("_pat_PATIENT_ID"),
                    F.to_date("SVC_DT_M").alias("_pat_SVC_DT_M"), "NEW_CKD_STAGE"))
    return (
        df.select(NPI_COL, "PATIENT_ID", SVC_COL)
        .withColumn("_SVC_DT_M", assign_cohort_month(F.col(SVC_COL), offset_days))
        .join(_pat, (F.col("PATIENT_ID") == F.col("_pat_PATIENT_ID")) &
                    (F.col("_SVC_DT_M") == F.col("_pat_SVC_DT_M")), how="left")
        .drop("_pat_PATIENT_ID", "_pat_SVC_DT_M")
        .withColumn("NEW_CKD_STAGE",
            F.when((F.col("NEW_CKD_STAGE") == 1), "NEW_TO_STAGE_1").otherwise(
            F.when((F.col("NEW_CKD_STAGE") == 2), "NEW_TO_STAGE_2").otherwise(
            F.when((F.col("NEW_CKD_STAGE") == 3), "NEW_TO_STAGE_3").otherwise(
            F.when((F.col("NEW_CKD_STAGE") == 4), "NEW_TO_STAGE_4").otherwise(
            F.when((F.col("NEW_CKD_STAGE") == 5), "NEW_TO_STAGE_5").otherwise("NO_NEW_STAGE"))))))
    )

df_dx_pat = add_patient_stage(df_dx, OFFSET_DAYS)

# ---------- Cohorts to process ----------
# Every cohort whose cutoff is on or before the last Dx service date (complete windows).
COHORT_LIST   = get_cohort_months(max_data_date=DATA_MAX_DT)
COHORT_BOUNDS = {m: get_cohort_bounds(m) for m in COHORT_LIST}
COHORT_IDS    = [COHORT_BOUNDS[m]["cohort_id"] for m in COHORT_LIST]
PREVIEW_COHORT = COHORT_MONTH if COHORT_MONTH in COHORT_LIST else COHORT_LIST[0]
cb_preview     = COHORT_BOUNDS[PREVIEW_COHORT]
print(f"Dx data through {DATA_MAX_DT} -> {len(COHORT_LIST)} complete cohorts: "
      f"{COHORT_LIST[0]} .. {COHORT_LIST[-1]} | preview cohort: {PREVIEW_COHORT}")

def dx_window_bounds(cb):
    """This notebook's lookbacks (1, 3, 6, 12) out of the config's window bounds."""
    return {w: cb["window_bounds"][w] for w in DX_LOOKBACK_WINDOWS}

# ---------- Fixed pivot values: same columns for every cohort ----------
# DX_STAGE labels present in the full Dx_03 (as the original pivot over all months);
# patient-stage labels are fixed by their definition (all present in the original output).
DX_STAGE_VALUES  = sorted(r[0] for r in add_dx_stage_label(df_dx_raw.select("DX_STAGE").distinct())
                          .distinct().collect() if r[0] is not None)
NEW_STAGE_VALUES = ["NEW_TO_STAGE_1", "NEW_TO_STAGE_2", "NEW_TO_STAGE_3", "NEW_TO_STAGE_4",
                    "NEW_TO_STAGE_5", "NO_NEW_STAGE"]
print(f"DX_STAGE values: {DX_STAGE_VALUES}")
print(f"NEW_CKD_STAGE values: {NEW_STAGE_VALUES}")

# ---------- Pre-filter to the widest window range over all cohorts ----------
_max_window = builtins.max(DX_LOOKBACK_WINDOWS)
_max_start  = COHORT_BOUNDS[COHORT_LIST[0]]["window_bounds"][_max_window]["start"]
_end        = COHORT_BOUNDS[COHORT_LIST[-1]]["window_bounds"][_max_window]["end"]
df_dx_filtered     = filter_to_window(df_dx, SVC_COL, _max_start, _end)
df_dx_pat_filtered = filter_to_window(df_dx_pat, SVC_COL, _max_start, _end)

print(f"Filtered Dx claims to ({_max_start}, {_end}]")
print(f"Row count: {df_dx_filtered.count():,} | with patient stage: {df_dx_pat_filtered.count():,}")

In [0]:
# ---- Display: raw Dx_03 claims (before any change) ----
# One row per Dx claim. DX_STAGE is still a number here (0 = unspecified, 1-5 = CKD stage).
_test_npis = get_test_npis()
_raw_view = df_dx_raw if _test_npis is None else df_dx_raw.filter(F.col(NPI_COL).isin(_test_npis))
print(f"Dx_03: {len(df_dx_raw.columns)} columns | rows for the NPIs in this run: {_raw_view.count():,}")
display(_raw_view.orderBy(NPI_COL, SVC_COL).limit(50))

In [0]:
# ---- Display: DX_STAGE number -> label, and the relabelled claims (df_dx) ----
# 1) How each raw stage number maps to its label, with claim counts
display(
    add_dx_stage_label(_raw_view.withColumn("DX_STAGE_RAW", F.col("DX_STAGE")))
    .groupBy("DX_STAGE_RAW", "DX_STAGE").count().orderBy("DX_STAGE_RAW")
)
# 2) df_dx: same claims, DX_STAGE now a label (used for RUNNING_ features = full history)
display(df_dx.orderBy(NPI_COL, SVC_COL).limit(50))

In [0]:
# ---- Display: patient CKD stage table (pat_new_ckd_stage) ----
# One row per patient per month. NEW_CKD_STAGE = stage the patient moved into that month
# (null or 0 = no change -> NO_NEW_STAGE). Shown only for patients of the NPIs in this run.
_pat_view = pat_new_ckd_stage.join(df_dx.select("PATIENT_ID").distinct(), on="PATIENT_ID", how="left_semi")
print(f"{_CKD_TABLE}: {_pat_view.count():,} rows for "
      f"{_pat_view.select('PATIENT_ID').distinct().count():,} patients of the NPIs in this run")
display(_pat_view.groupBy("NEW_CKD_STAGE").count().orderBy("NEW_CKD_STAGE"))
display(_pat_view.orderBy("PATIENT_ID", "SVC_DT_M").limit(50))

In [0]:
# ---- Display: claims after the patient-stage join, and the pre-filtered claim table ----
# 1) df_dx_pat_filtered: claims where the patient moved to a new stage in the claim's cohort month
#    (_SVC_DT_M = cohort month of the claim, offset rule applied)
display(df_dx_pat_filtered.filter(F.col("NEW_CKD_STAGE") != "NO_NEW_STAGE")
        .orderBy(NPI_COL, SVC_COL).limit(50))
# 2) df_dx_filtered: all claim columns, restricted to the widest window range of all cohorts
#    (used for the _IN_LAST_N_MONTH features)
display(df_dx_filtered.orderBy(NPI_COL, SVC_COL).limit(50))

In [0]:
display(df_dx_pat_filtered.orderBy(NPI_COL, SVC_COL).limit(100))

In [0]:
# Full history (used for RUNNING_) and windowed range (used for _IN_LAST_N_MONTH)
display(df_dx.agg(F.min(SVC_COL).alias("min_svc_dt"), F.max(SVC_COL).alias("max_svc_dt"), F.count(F.lit(1)).alias("rows")))
display(df_dx_filtered.agg(F.min(SVC_COL).alias("min_svc_dt"), F.max(SVC_COL).alias("max_svc_dt"), F.count(F.lit(1)).alias("rows")))

# Claims per patient-stage label (left join keeps every claim: counts must equal the Dx claims)
display(df_dx_pat_filtered.groupBy("NEW_CKD_STAGE").count().orderBy("NEW_CKD_STAGE"))

# Cutoff and window start (exclusive) for every cohort; every window ends at the cutoff
display(spark.createDataFrame(
    [(COHORT_BOUNDS[m]["cohort_id"], COHORT_BOUNDS[m]["cutoff"].strftime("%Y-%m-%d"),
      *[COHORT_BOUNDS[m]["window_bounds"][w]["start"] for w in DX_LOOKBACK_WINDOWS])
     for m in COHORT_LIST],
    "COHORT_ID string, CUTOFF_DATE string, " +
    ", ".join(f"START_{w}M string" for w in DX_LOOKBACK_WINDOWS)
))

## 2. Create Features
Each step is a function of one cohort's bounds (`cb = get_cohort_bounds(month)`). The preview at the end of each cell runs it on `cb_preview` only; section 4 runs every cohort.

In [0]:
# Features are built per cohort on NPI; COHORT_ID is added afterwards, so the
# output grain is one row per NPI per cohort (was NPI x SVC_DT_M)
index_cols = [NPI_COL]

In [0]:
# ---- 2.1 Diagnosis Category + 2.2 Patient CKD Stage + total, per lookback window (one cohort) ----
# Same pivots and counts as the original, but each window is a date range from the cohort's
# window_bounds instead of a row-window over monthly rows. Column names unchanged:
#   TOTAL_DX_CLAIMS_{STAGE}_IN_LAST_{N}_MONTH, TOTAL_DX_CLAIMS_WITH_PATIENT_{NEW_STAGE}_IN_LAST_{N}_MONTH,
#   TOTAL_DX_CLAIMS_IN_LAST_{N}_MONTH. Fixed pivot values -> every cohort gets the same columns.

def build_window_features(df, df_pat, cb):
    wb = dx_window_bounds(cb)
    df_category = build_feature_group(
        df, NPI_COL, SVC_COL, pivot_col="DX_STAGE", prefix="TOTAL_DX_CLAIMS_",
        window_bounds=wb, pivot_values=DX_STAGE_VALUES)
    df_patient_stage = build_feature_group(
        df_pat, NPI_COL, SVC_COL, pivot_col="NEW_CKD_STAGE", prefix="TOTAL_DX_CLAIMS_WITH_PATIENT_",
        window_bounds=wb, pivot_values=NEW_STAGE_VALUES)
    # total claims (denominator for PCT)
    df_total = build_total_col(df, NPI_COL, SVC_COL, wb, total_name="TOTAL_DX_CLAIMS")
    return df_category, df_patient_stage, df_total


def _display_with_bounds(df, cb, n=100):
    """Display-only: COHORT_ID plus a WINDOW_{N}M column with its (start, end] date range
    in front of each window's features. The feature DataFrames are not changed."""
    _df_cols = df.columns
    _cols = [F.col(NPI_COL), F.lit(cb["cohort_id"]).alias("COHORT_ID")]
    for w in DX_LOOKBACK_WINDOWS:
        b = cb["window_bounds"][w]
        _cols.append(F.lit(f"({b['start']}, {b['end']}]").alias(f"WINDOW_{w}M"))
        _cols += [F.col(c) for c in _df_cols if c.endswith(f"_IN_LAST_{w}_MONTH")]
    display(df.select(*_cols).orderBy(NPI_COL).limit(n))


# ---- Preview on one cohort (all cohorts are built and written in section 4) ----
df_category, df_patient_stage, df_total = build_window_features(df_dx_filtered, df_dx_pat_filtered, cb_preview)
_display_with_bounds(df_category, cb_preview)
_display_with_bounds(df_patient_stage, cb_preview)
_display_with_bounds(df_total, cb_preview)
print(f"Preview {cb_preview['cohort_id']} | Dx stage cols: {len(df_category.columns) - 1} | "
      f"Patient stage cols: {len(df_patient_stage.columns) - 1} | Total cols: {len(df_total.columns) - 1}")

In [0]:
# ---- RUNNING_ features (full Dx history through the cohort cutoff) ----
# Reproduces step3_lag's unbounded-preceding RUNNING_ columns: every Dx claim from the
# start of Dx_03 history up to and including the cohort cutoff.
# Names unchanged: RUNNING_TOTAL_DX_CLAIMS, RUNNING_TOTAL_DX_CLAIMS_{STAGE},
# RUNNING_TOTAL_DX_CLAIMS_WITH_PATIENT_{NEW_STAGE}

def build_running_features(df, df_pat, cb):
    cutoff_lit = F.to_date(F.lit(cb["cutoff"].strftime("%Y-%m-%d")))
    df_run     = df.filter(F.col(SVC_COL) <= cutoff_lit)
    df_run_pat = df_pat.filter(F.col(SVC_COL) <= cutoff_lit)

    df_running_category = pivoted_counts(df_run, NPI_COL, "DX_STAGE",
                                         "RUNNING_TOTAL_DX_CLAIMS_", "", values=DX_STAGE_VALUES)
    df_running_patient  = pivoted_counts(df_run_pat, NPI_COL, "NEW_CKD_STAGE",
                                         "RUNNING_TOTAL_DX_CLAIMS_WITH_PATIENT_", "", values=NEW_STAGE_VALUES)
    df_running_total    = df_run.groupBy(NPI_COL).agg(F.count(F.lit(1)).alias("RUNNING_TOTAL_DX_CLAIMS"))
    return (
        df_running_total
        .join(df_running_category, on=NPI_COL, how="left")
        .join(df_running_patient, on=NPI_COL, how="left")
        .na.fill(0)
    )


# ---- Preview on one cohort ----
df_running = build_running_features(df_dx, df_dx_pat, cb_preview)
print(f"Preview {cb_preview['cohort_id']} | Running universe: {df_running.count():,} NPIs | "
      f"RUNNING_ cols: {len([c for c in df_running.columns if c.startswith('RUNNING_')])}")
display(df_running.orderBy(NPI_COL))

In [0]:
# ---- 2.3 Combine features — one row per NPI ----
# NPI universe = NPIs with Dx claims in the widest window + NPIs with older history only,
# so their RUNNING_ values are kept (same approach as NB01/NB02).
def combine_features(df, df_category, df_patient_stage, df_total, df_running, cb):
    _wb = cb["window_bounds"][builtins.max(DX_LOOKBACK_WINDOWS)]
    npi_universe = filter_to_window(df, SVC_COL, _wb["start"], _wb["end"]).select(NPI_COL).distinct()
    npi_universe_wide = npi_universe.union(df_running.select(NPI_COL)).distinct()
    return (
        npi_universe_wide
        .join(df_total, how='left', on=index_cols)
        .join(df_category, how='left', on=index_cols)
        .join(df_patient_stage, how='left', on=index_cols)
        .join(df_running, how='left', on=index_cols)
        .na.fill(value=0)
    )


# ---- Preview on one cohort ----
df_spine = combine_features(df_dx_filtered, df_category, df_patient_stage, df_total, df_running, cb_preview)
print(f"Preview {cb_preview['cohort_id']} | Combined spine: {df_spine.count():,} NPIs, {len(df_spine.columns)} columns")

In [0]:
# ---- Display: combined spine (preview cohort, one row per NPI) ----
# Window totals, Dx-stage counts, patient-stage counts and RUNNING_ columns side by side; missing = 0.
# Run right after the Combine cell (the PCT cell below adds more columns to df_spine).
display(df_spine.orderBy(NPI_COL))

## 3. Create Pct Features

In [0]:
# ---- PCT features: each feature / TOTAL_DX_CLAIMS in the same window ----
# Same definition as the original (PCT_ + feature name), without exec(). The original base-month
# PCT_ columns are the _IN_LAST_1_MONTH ones here. Safe division as in NB01/NB02: 0.0 when the
# window total is 0. RUNNING_ columns get no PCT_ (as in the original).
total_col = "TOTAL_DX_CLAIMS"

def add_pct_features(df_spine):
    _spine_cols = df_spine.columns
    pct_features = []
    for i in DX_LOOKBACK_WINDOWS:
        denominator = f"{total_col}_IN_LAST_{i}_MONTH"
        features_i = [c for c in _spine_cols
                      if c.endswith(f"_IN_LAST_{i}_MONTH") and c != denominator]
        pct_features += [
            F.when(F.col(denominator) != 0, F.col(feature) / F.col(denominator))
             .otherwise(F.lit(0.0))
             .alias('PCT_' + feature)
            for feature in features_i
        ]
    return df_spine.select("*", *pct_features)


# ---- Preview on one cohort ----
df_spine = add_pct_features(df_spine)
print(f"Preview {cb_preview['cohort_id']} | After PCT: {len(df_spine.columns)} columns")

In [0]:
# ---- Display: PCT_ columns next to their window totals (preview cohort) ----
_tot_cols = [f"TOTAL_DX_CLAIMS_IN_LAST_{w}_MONTH" for w in DX_LOOKBACK_WINDOWS]
_pct_cols = [c for c in df_spine.columns if c.startswith("PCT_")]
print(f"{len(_pct_cols)} PCT_ columns")
display(df_spine.select(NPI_COL, *_tot_cols, *_pct_cols).orderBy(NPI_COL))

In [0]:
# ---- Add cohort metadata (replaces the hardcoded SVC_DT_M >= '2022-01-01' filter) ----
def add_cohort_metadata(df_spine, cb):
    return (
        df_spine
        .withColumn("COHORT_ID", F.lit(cb["cohort_id"]))
        .withColumn("COHORT_MONTH", F.lit(cb["cohort_month"]))
        .withColumn("OFFSET_DAYS", F.lit(cb["offset_days"]))
        .withColumn("CUTOFF_DATE", F.lit(cb["cutoff"].strftime("%Y-%m-%d")))
        .withColumn("SVC_DT_M", F.to_timestamp(F.lit(f"{cb['cohort_month']}-01")))
    )


# ---- Preview on one cohort ----
claim_diagnosis_monthly = add_cohort_metadata(df_spine, cb_preview)
print(f"Preview {cb_preview['cohort_id']} | Final output columns: {len(claim_diagnosis_monthly.columns)}")
display(claim_diagnosis_monthly.orderBy(NPI_COL))

## 4. Build and Write All Cohorts
Runs the same steps as sections 2–3 (`build_window_features` → `build_running_features` → `combine_features` → `add_pct_features` → `add_cohort_metadata`) for every cohort in `COHORT_LIST` and writes one `COHORT_ID` partition per cohort (`replaceWhere`), then the provider-level copy.

In [0]:
# New output path, partitioned by COHORT_ID. Wenjie's provider_claim_diagnosis_monthly is NOT touched.
import time

OUTPUT_PATH = (
    "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/"
    "kerendia_nba_3_volume/hcp_level_model/ops_pipeline/notebook_output/"
    "claim_diagnosis_monthly_cohort"
)

def build_cohort(cb, df=None, df_pat=None):
    """All feature steps for one cohort (same functions as the preview above).
    df / df_pat: claim bases (default: the run's pre-filtered windows); RUNNING_ uses the same
    bases when given, else the full history (df_dx / df_dx_pat)."""
    df_w, df_w_pat = (df_dx_filtered, df_dx_pat_filtered) if df is None else (df, df_pat)
    df_r, df_r_pat = (df_dx, df_dx_pat) if df is None else (df, df_pat)
    df_category, df_patient_stage, df_total = build_window_features(df_w, df_w_pat, cb)
    df_running = build_running_features(df_r, df_r_pat, cb)
    df_spine   = combine_features(df_w, df_category, df_patient_stage, df_total, df_running, cb)
    df_spine   = add_pct_features(df_spine)
    return add_cohort_metadata(df_spine, cb).withColumn("PULL_DT", F.current_date())


_t_all = time.time()
for m in COHORT_LIST:
    cb = COHORT_BOUNDS[m]
    _t0 = time.time()
    build_cohort(cb).write \
        .format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true") \
        .option("replaceWhere", f"COHORT_ID = '{cb['cohort_id']}'") \
        .partitionBy("COHORT_ID") \
        .save(OUTPUT_PATH)
    print(f"{cb['cohort_id']}: written (cutoff {cb['cutoff']:%Y-%m-%d}) in {time.time() - _t0:,.0f}s")

print(f"\n{len(COHORT_LIST)} cohorts written to {OUTPUT_PATH} in {(time.time() - _t_all) / 60:,.1f} min")

In [0]:
# Read the written table directly (independent of the preview variables)
df_all_cohorts = spark.read.format("delta").load(OUTPUT_PATH)

# ---- Completeness checks ----
_ids_in_table  = sorted(r[0] for r in df_all_cohorts.select("COHORT_ID").distinct().collect())
_missing       = sorted(set(COHORT_IDS) - set(_ids_in_table))
_extra         = sorted(set(_ids_in_table) - set(COHORT_IDS))
df_all_cohorts = df_all_cohorts.filter(F.col("COHORT_ID").isin(COHORT_IDS))
_n_rows        = df_all_cohorts.count()
_n_keys        = df_all_cohorts.select(NPI_COL, "COHORT_ID").distinct().count()
_n_npis        = df_all_cohorts.select(NPI_COL).distinct().count()
_nulls         = df_all_cohorts.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_all_cohorts.columns]).first()
_null_cells    = builtins.sum(v or 0 for v in _nulls)

print(f"Cohorts expected: {len(COHORT_IDS)} | in table: {len(_ids_in_table)} | "
      f"missing: {_missing or 'none'} | extra (not in this run): {_extra or 'none'}")
print(f"Rows: {_n_rows:,} | distinct (NPI, COHORT_ID): {_n_keys:,} | duplicates: {_n_rows - _n_keys} | "
      f"NPIs: {_n_npis} | Columns: {len(df_all_cohorts.columns)} | null cells: {_null_cells:,}")
display(df_all_cohorts.groupBy("COHORT_ID", "CUTOFF_DATE").count().orderBy("COHORT_ID"))

# ---- Full table: every NPI x cohort, all columns (keys and dates first) ----
_key_cols  = [NPI_COL, "COHORT_ID", "COHORT_MONTH", "OFFSET_DAYS", "CUTOFF_DATE", "SVC_DT_M", "PULL_DT"]
_feat_cols = [c for c in df_all_cohorts.columns if c not in _key_cols]
display(df_all_cohorts.select(*_key_cols, *_feat_cols).orderBy(NPI_COL, "COHORT_ID"))

In [0]:
# Same mapping as the original (NPI -> PROVIDER_ID, NPI dropped), on a separate variable so the
# NPI-level table above stays available. Wenjie's provider_claim_diagnosis_monthly is NOT touched.
hcp_npi_mapping = spark.read.format('delta').load(_QUERY_OUTPUT + "kerendia_nba_3_0_npi_bhid_mapping")
hcp_npi_mapping_df = hcp_npi_mapping.select('NPI_NUMBER', 'PROVIDER_ID').distinct()

provider_claim_diagnosis_monthly = (
    spark.read.format("delta").load(OUTPUT_PATH)
    .filter(F.col("COHORT_ID").isin(COHORT_IDS))
    .join(hcp_npi_mapping_df, on='NPI_NUMBER', how='left')
    .drop('NPI_NUMBER')
)

PROVIDER_OUTPUT_PATH = (
    "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/"
    "kerendia_nba_3_volume/hcp_level_model/ops_pipeline/notebook_output/"
    "provider_claim_diagnosis_monthly_cohort"
)
provider_claim_diagnosis_monthly.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .option("replaceWhere", "COHORT_ID IN (" + ", ".join(f"'{c}'" for c in COHORT_IDS) + ")") \
    .partitionBy("COHORT_ID") \
    .save(PROVIDER_OUTPUT_PATH)

_prov = spark.read.format("delta").load(PROVIDER_OUTPUT_PATH).filter(F.col("COHORT_ID").isin(COHORT_IDS))
print(f"Written to {PROVIDER_OUTPUT_PATH} ({len(COHORT_IDS)} cohorts) | rows: {_prov.count():,} | "
      f"null PROVIDER_ID: {_prov.filter(F.col('PROVIDER_ID').isNull()).count():,} | "
      f"duplicate (PROVIDER_ID, COHORT_ID): {_prov.count() - _prov.select('PROVIDER_ID', 'COHORT_ID').distinct().count():,}")

In [0]:
# ---- Display: NPI -> PROVIDER_ID mapping and the provider-level output ----
# 1) Mapping table (raw, includes BH_ID) for the NPIs in this run
_map_view = hcp_npi_mapping if _test_npis is None else hcp_npi_mapping.filter(F.col(NPI_COL).isin(_test_npis))
display(_map_view.orderBy(NPI_COL))
# 2) provider_claim_diagnosis_monthly_cohort: same rows as the NPI table, NPI replaced by PROVIDER_ID
_first = ["PROVIDER_ID", "COHORT_ID", "CUTOFF_DATE"]
display(_prov.select(*_first, *[c for c in _prov.columns if c not in _first])
        .orderBy("PROVIDER_ID", "COHORT_ID"))

## 5. Validation

1. **Offset = 0 parity (every cohort)**: rebuild the features at offset 0 for the test NPIs with the same functions and the same inputs as the original (3.0 `pat_new_ckd_stage`, patient-stage month at offset 0), map NPI → PROVIDER_ID, and compare every column with Wenjie's `provider_claim_diagnosis_monthly` for each month (the original only has the provider-level output). Original base-month columns (no suffix) are compared with `_IN_LAST_1_MONTH`; original nulls are treated as 0. Providers only in the original for a month (no Dx history yet) must be all zeros. Result 2026-10-21: 19 months x 10 providers x 127 columns, 0 mismatches. (With the 3.1 validation patient-stage table only `WITH_PATIENT_*` columns differ, which is an input difference, not a logic difference.)
2. **HCP trace (every cohort)**: for the busiest test NPI, recount raw Dx claims per cohort and window in pandas (independent of Spark), including the patient-stage join, and compare with the written features. Result 2026-10-21: NPI 1508230566, 19 cohorts, 1,140 checks, 0 mismatches.

In [0]:
# ---- Validation 1: offset = 0 parity vs original provider_claim_diagnosis_monthly, every cohort ----
import numpy as np
import pandas as pd

_npis = get_test_npis()
if _npis is None:
    print("Production mode — parity check skipped (runs on the test NPIs only)")
else:
    _dx0     = df_dx.filter(F.col(NPI_COL).isin(_npis))
    # Same inputs as the original: the 3.0 pat_new_ckd_stage (calendar months), restricted to the
    # test NPIs' patients and materialised once (small). Patient-stage month at offset 0.
    _pids   = [r[0] for r in _dx0.select("PATIENT_ID").distinct().collect()]
    _pat30  = spark.createDataFrame(
        spark.read.format("delta").load(_QUERY_OUTPUT + "pat_new_ckd_stage")
        .filter(F.col("PATIENT_ID").isin(_pids))
        .select("PATIENT_ID", F.to_date("SVC_DT_M").alias("SVC_DT_M"),
                F.col("NEW_CKD_STAGE").cast("int").alias("NEW_CKD_STAGE")).toPandas())
    _dx0_pat = add_patient_stage(_dx0, 0, pat_df=_pat30)
    _map_pd  = hcp_npi_mapping_df.filter(F.col(NPI_COL).isin(_npis)).toPandas()
    _npi_to_prov = dict(zip(_map_pd[NPI_COL], _map_pd["PROVIDER_ID"]))

    _orig_all = (
        spark.read.format("delta").load(
            "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/"
            "kerendia_nba_3_volume/hcp_level_model/ops_pipeline/notebook_output/provider_claim_diagnosis_monthly")
        .filter(F.col("PROVIDER_ID").isin(list(_npi_to_prov.values())))
        .withColumn("_M", F.date_format("SVC_DT_M", "yyyy-MM"))
    )
    _orig_months = {r[0] for r in _orig_all.select("_M").distinct().collect()}
    # Same cohort range, with the complete-data rule evaluated at offset 0
    PARITY_MONTHS = [m for m in get_cohort_months(max_data_date=DATA_MAX_DT, offset_days=0) if m in _orig_months]
    pd_orig_all = _orig_all.filter(F.col("_M").isin(PARITY_MONTHS)).toPandas()

    _skip = {"SVC_DT_M", "PULL_DT", "_M", "PROVIDER_ID"}
    _orig_cols = [c for c in pd_orig_all.columns if c not in _skip]
    # Original column -> cohort-based column (base-month columns map to _IN_LAST_1_MONTH)
    _map = {oc: (oc if ("_IN_LAST_" in oc or oc.startswith("RUNNING_")) else f"{oc}_IN_LAST_1_MONTH")
            for oc in _orig_cols}

    def _num(df):
        return df.apply(pd.to_numeric, errors="coerce").fillna(0).astype(float)

    summary, mismatches = [], []
    for m in PARITY_MONTHS:
        _t0 = time.time()
        cb0 = get_cohort_bounds(m, offset_days=0)
        pd_new = build_cohort(cb0, df=_dx0, df_pat=_dx0_pat).toPandas()
        pd_new["PROVIDER_ID"] = pd_new[NPI_COL].map(_npi_to_prov)
        pd_new  = pd_new.set_index("PROVIDER_ID")
        pd_orig = pd_orig_all[pd_orig_all["_M"] == m].set_index("PROVIDER_ID")

        common    = sorted(set(pd_new.index) & set(pd_orig.index))
        only_orig = sorted(set(pd_orig.index) - set(pd_new.index))
        only_new  = sorted(set(pd_new.index) - set(pd_orig.index))
        missing_new_cols = sorted(nc for nc in _map.values() if nc not in pd_new.columns)

        o = _num(pd_orig.loc[common, list(_map.keys())])
        n = _num(pd_new.reindex(columns=list(_map.values())).loc[common])
        bad = ~np.isclose(o.values, n.values, rtol=1e-9, atol=1e-9)
        _oc = list(_map.keys())
        for i, j in list(zip(*np.where(bad)))[:50]:
            mismatches.append({"cohort": cb0["cohort_id"], "PROVIDER_ID": common[i], "original_col": _oc[j],
                               "new_col": _map[_oc[j]], "original": o.values[i, j], "new": n.values[i, j]})

        # Providers only in the original (no Dx history yet at that month) must be all zeros
        nonzero_only_orig = (int((_num(pd_orig.loc[only_orig, _orig_cols]) != 0).any(axis=1).sum())
                             if only_orig else 0)

        summary.append({"cohort_offset0": cb0["cohort_id"], "cutoff": f"{cb0['cutoff']:%Y-%m-%d}",
                        "providers_common": len(common), "providers_only_original": len(only_orig),
                        "only_original_nonzero": nonzero_only_orig, "providers_only_new": len(only_new),
                        "columns_compared": len(_map), "new_cols_missing": len(missing_new_cols),
                        "mismatching_cells": int(bad.sum()), "seconds": builtins.round(time.time() - _t0)})
        print(f"{cb0['cohort_id']}: {int(bad.sum())} mismatching cells ({time.time() - _t0:,.0f}s)")

    df_parity = pd.DataFrame(summary)
    display(df_parity)
    if mismatches:
        _mm = pd.DataFrame(mismatches)
        _mm["group"] = np.where(_mm["original_col"].str.contains("WITH_PATIENT"), "patient_stage", "dx_stage_or_total")
        print("Mismatches by column group (first 50 per cohort):")
        display(_mm.groupby("group").size().reset_index(name="cells"))
    _ok = (df_parity[["mismatching_cells", "only_original_nonzero", "providers_only_new",
                      "new_cols_missing"]].values.sum() == 0)
    print(f"Cohorts compared: {len(df_parity)} | columns: {len(_map)} | "
          f"mismatching cells: {df_parity['mismatching_cells'].sum()} | {'PASS' if _ok else 'CHECK MISMATCHES'}")
    if mismatches:
        display(pd.DataFrame(mismatches))

In [0]:
# ---- Display: what the offset-0 parity check compares ----
if _npis is not None:
    # 1) Wenjie's original provider table: one row per provider per calendar month (SVC_DT_M)
    display(_orig_all.drop("_M").orderBy("PROVIDER_ID", "SVC_DT_M").limit(100))
    # 2) 3.0 patient-stage rows used for the parity run (same input as the original)
    display(_pat30.orderBy("PATIENT_ID", "SVC_DT_M").limit(50))
    # 3) Side by side for one provider in the last compared month: original column -> new column
    _m = PARITY_MONTHS[-1]
    _p = pd_new.loc[common, "TOTAL_DX_CLAIMS_IN_LAST_12_MONTH"].astype(float).idxmax()
    _side = pd.DataFrame({"original_col": list(_map.keys()), "new_col": list(_map.values())})
    _side["original"] = [float(pd_orig.at[_p, oc]) if pd.notna(pd_orig.at[_p, oc]) else np.nan for oc in _map]
    _side["new"] = [float(pd_new.at[_p, nc]) for nc in _map.values()]
    print(f"Provider {_p}, month {_m} (offset 0): original NaN = null in the original (compared as 0)")
    display(_side)

In [0]:
# ---- Validation 2: HCP trace — raw Dx claims recounted in pandas vs written features, every cohort ----
import pandas as pd

TRACE_NPI = df_dx_filtered.groupBy(NPI_COL).count().orderBy(F.desc("count")).first()[NPI_COL]

_raw = (df_dx_raw.filter(F.col(NPI_COL) == TRACE_NPI)
        .select("PATIENT_ID", SVC_COL, F.col("DX_STAGE").cast("int").alias("DX_STAGE")).toPandas())
_raw[SVC_COL] = pd.to_datetime(_raw[SVC_COL])
_raw["STAGE"] = _raw["DX_STAGE"].map({1: "STAGE_1", 2: "STAGE_2", 3: "STAGE_3", 4: "STAGE_4",
                                      5: "STAGE_5"}).fillna("STAGE_UNSPECIFIED")

# Patient-stage join in pandas: claim month = same rule as assign_cohort_month (offset-th day starts new month)
_off = OFFSET_DAYS
_raw["_M"] = ((_raw[SVC_COL] - pd.Timedelta(days=_off - 1)) if _off > 0 else _raw[SVC_COL]).dt.to_period("M").dt.to_timestamp()
_pat = (pat_new_ckd_stage.filter(F.col("PATIENT_ID").isin(_raw["PATIENT_ID"].unique().tolist()))
        .select("PATIENT_ID", F.to_date("SVC_DT_M").alias("_M"), F.col("NEW_CKD_STAGE").cast("int").alias("NEW"))
        .toPandas())
_pat["_M"] = pd.to_datetime(_pat["_M"])
_raw_pat = _raw.merge(_pat, on=["PATIENT_ID", "_M"], how="left")
_raw_pat["NEW_STAGE"] = _raw_pat["NEW"].map({1: "NEW_TO_STAGE_1", 2: "NEW_TO_STAGE_2", 3: "NEW_TO_STAGE_3",
                                             4: "NEW_TO_STAGE_4", 5: "NEW_TO_STAGE_5"}).fillna("NO_NEW_STAGE")

_out = (spark.read.format("delta").load(OUTPUT_PATH)
        .filter((F.col(NPI_COL) == TRACE_NPI) & F.col("COHORT_ID").isin(COHORT_IDS))
        .toPandas().set_index("COHORT_ID"))

def _v(d, k):
    x = d.get(k, 0)
    return 0 if pd.isna(x) else int(x)

def _in(df, start, end):
    return df[(df[SVC_COL] > pd.Timestamp(start)) & (df[SVC_COL] <= pd.Timestamp(end))]

rows, bad = [], []
for m in COHORT_LIST:
    cb, cid = COHORT_BOUNDS[m], COHORT_BOUNDS[m]["cohort_id"]
    feat = _out.loc[cid].to_dict() if cid in _out.index else {}
    row = {"COHORT_ID": cid, "CUTOFF_DATE": f"{cb['cutoff']:%Y-%m-%d}"}
    expected = {}
    for w in DX_LOOKBACK_WINDOWS:
        b, sfx = cb["window_bounds"][w], f"_IN_LAST_{w}_MONTH"
        sel, sel_p = _in(_raw, b["start"], b["end"]), _in(_raw_pat, b["start"], b["end"])
        expected[f"TOTAL_DX_CLAIMS{sfx}"] = len(sel)
        for v in DX_STAGE_VALUES:
            expected[f"TOTAL_DX_CLAIMS_{v}{sfx}"] = int((sel["STAGE"] == v).sum())
        for v in NEW_STAGE_VALUES:
            expected[f"TOTAL_DX_CLAIMS_WITH_PATIENT_{v}{sfx}"] = int((sel_p["NEW_STAGE"] == v).sum())
        row[f"{w}M_raw"], row[f"{w}M_feature"] = len(sel), _v(feat, f"TOTAL_DX_CLAIMS{sfx}")
    run, run_p = _raw[_raw[SVC_COL] <= pd.Timestamp(cb["cutoff"])], _raw_pat[_raw_pat[SVC_COL] <= pd.Timestamp(cb["cutoff"])]
    expected["RUNNING_TOTAL_DX_CLAIMS"] = len(run)
    for v in DX_STAGE_VALUES:
        expected[f"RUNNING_TOTAL_DX_CLAIMS_{v}"] = int((run["STAGE"] == v).sum())
    for v in NEW_STAGE_VALUES:
        expected[f"RUNNING_TOTAL_DX_CLAIMS_WITH_PATIENT_{v}"] = int((run_p["NEW_STAGE"] == v).sum())
    n_bad = 0
    for fname, raw_n in expected.items():
        if _v(feat, fname) != raw_n:
            n_bad += 1
            bad.append({"COHORT_ID": cid, "feature": fname, "raw": raw_n, "feature_value": _v(feat, fname)})
    row.update({"RUNNING_raw": len(run), "RUNNING_feature": _v(feat, "RUNNING_TOTAL_DX_CLAIMS"),
                "checks": len(expected), "mismatches": n_bad})
    rows.append(row)

trace = pd.DataFrame(rows)
print(f"Trace NPI: {TRACE_NPI} | cohorts: {len(trace)} | checks: {trace['checks'].sum():,} | "
      f"mismatches: {trace['mismatches'].sum()}")
display(trace)
if bad:
    display(pd.DataFrame(bad).head(50))

In [0]:
# ---- Display: raw claims used by the trace (pandas, independent of the Spark features) ----
# _M = claim's cohort month (offset rule), STAGE = Dx-stage label, NEW_STAGE = patient-stage label
print(f"Trace NPI {TRACE_NPI}: {len(_raw_pat):,} claims in full history")
display(_raw_pat.sort_values(SVC_COL).tail(50))
display(_raw_pat.groupby(["STAGE", "NEW_STAGE"]).size().reset_index(name="claims"))